<a href="https://colab.research.google.com/github/JoseAlberto88/Hugging-Face-Text-Classification/blob/main/huggingface_text_classification_tutorial_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Learning Hugging Face Text Classification Tutorial

* Resources notebook: https://www.learnhuggingface.com/notebooks/hugging_face_text_classification_tutorial
* Setup steps: https://www.learnhuggingface.com/extras/setup

**Note** A GPU is needed on Google Colab, go to Runtime -> Change runtime -> Hardware accelerator -> GPU.

### Import necessary libraries

In [1]:
import transformers

In [1]:
# Install dependencies (this is mostly for Google Colab)
try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio
  import datasets, evaluate, accelerate
  import gradio as gr

import random

import numpy as np
import pandas as pd

import torch
import transformers

print(f"Using transformers version: {transformers.__version__}")
print(f"Using torch version: {torch.__version__}" )
print(f"Using datasets version: {datasets.__version__}")


Using transformers version: 5.15.0
Using torch version: 2.11.0+cu128
Using datasets version: 5.0.1


## 3. Getting a dataset

Building food not food text classification model: need food not food text dataset.

In [20]:
from datasets import load_dataset

dataset = load_dataset(path="mrdbourke/learn_hf_food_not_food_image_captions")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 250
    })
})

In [21]:
# What features are there ?

dataset.column_names

{'train': ['text', 'label']}

In [22]:
dataset["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 250
})

In [23]:
dataset["train"][0]

{'text': 'Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
 'label': 'food'}

### Import random samples


In [24]:
import random

random_indexs = random.sample(range(len(dataset["train"])), 5)

random_samples = dataset["train"][random_indexs]

print(f"[INFO] Random samples from dataset:\n")
for text, label in zip(random_samples["text"], random_samples["label"]):
  print(f"Text: {text} | Label: {label}")

[INFO] Random samples from dataset:

Text: Robust beef curry in a hearty bowl, simmered with a medley of spices, tomatoes, and onions, garnished with chopped green onions. | Label: food
Text: Lawn mower stored in a shed | Label: not_food
Text: Parsnips in a bowl, sprinkled with thyme and served with a side of honey for a tasty, unique snack. | Label: food
Text: Set of keys hanging on a hook by the door | Label: not_food
Text: Celery in a bowl, served with a side of peanut butter and a sprinkle of raisins for a classic, tasty snack. | Label: food


In [25]:
# Get unique label values
dataset["train"].unique("label")

['food', 'not_food']

In [26]:
# Check the count of each label
from collections import Counter

Counter(dataset["train"]["label"])

Counter({'food': 125, 'not_food': 125})

In [27]:
# Turn our dataset into a DataFrame and get a random sample
food_not_food_df = pd.DataFrame(dataset["train"])
food_not_food_df.sample(7)

,text,label
195,Bicycle leaning casually against a wall,not_food
77,Silverware organizer keeping cutlery tidy in a...,not_food
219,Two dogs laying on a rug,not_food
232,"Beets in a bowl, sprinkled with goat cheese an...",food
47,Wooden hanger holding clothes on a rack,not_food
75,"Sushi with a spicy kick, featuring jalapeno pe...",food
94,Hearty pumpkin curry with toasted pumpkin seed...,food


In [28]:
food_not_food_df["label"].value_counts()

,count
label,
food,125
not_food,125


## 4. Preparing data for text classification

We want to:

1. Tokenize our text -> turn our text into numbers (this goes for labels as well).
2. Create a train/test split -> want to train our model on the training split and want to evaluate our model on the test split.

In [29]:
# Create a mapping for labels to numeric value

id2label = {0: "not_food", 1: "food"}
label2id = {"not_food" : 0, "food" : 1}

print(id2label)
print(label2id)

{0: 'not_food', 1: 'food'}
{'not_food': 0, 'food': 1}


In [30]:
# Create mappings programmatically from dataset
id2label = {idx : label for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
id2label

{0: 'not_food', 1: 'food'}

In [32]:
label2id = {label : idx for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
label2id

{'not_food': 0, 'food': 1}

In [33]:
# turn labels into 0 or 1

def map_labels_to_number(example):
  example["label"] = label2id[example["label"]]
  return example

example_sample = {"text" : "This is a sentence about my favprite food: honey", "label": "food"}

# Test our function
map_labels_to_number(example_sample)

{'text': 'This is a sentence about my favprite food: honey', 'label': 1}

In [34]:
# Map our dataset labels to numbers (the whole thing)
# We do this with dataset.map()  - https://huggingface.co/docs/datasets/process#map
dataset = dataset["train"].map(map_labels_to_number)
dataset[:5]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

{'text': ['Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
  'Set of books stacked on a desk',
  'Watching TV together, a family has their dog stretched out on the floor',
  'Wooden dresser with a mirror reflecting the room',
  'Lawn mower stored in a shed'],
 'label': [1, 0, 0, 0, 0]}

In [35]:
# Shuffle data and look at more 5 random examples
dataset.shuffle()[:5]

{'text': ['Set of muffin tins stacked together',
  'Tangy fish curry bowl, featuring delicate fish pieces in a zesty sauce made with tamarind and curry leaves, ideal for a light meal.',
  'Set of board games stacked on a shelf',
  'Green beans in a bowl, sprinkled with almonds and served with a side of lemon vinaigrette for a healthy, tasty dish.',
  'Set of binoculars placed on a table'],
 'label': [0, 1, 0, 1, 0]}